In [21]:
pip install langchain_community langchain_openai faiss-cpu pypdf

Note: you may need to restart the kernel to use updated packages.


In [1]:
!pip install python-dotenv
import os 
from dotenv import load_dotenv

# .env 파일 로드
load_dotenv()

# 환경 변수 가져오기
API_KEY = os.getenv("API_KEY")

In [9]:
#### chatGPT RAG 활용
import os
import openai
from langchain.chains import AnalyzeDocumentChain
from langchain.chains.question_answering import load_qa_chain
from langchain.chat_models import ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains import RetrievalQAWithSourcesChain, LLMChain, StuffDocumentsChain
# from langchain.retrievers import EmbeddingRetriever
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain.document_loaders import DirectoryLoader, TextLoader
from langchain.docstore.document import Document
import numpy as np
####
from dotenv import load_dotenv
import json
import faiss
from openai import OpenAI


#### api 키 설정
api_key = API_KEY
os.environ['OPENAI_API_KEY'] = api_key
####

client = OpenAI(
    api_key=api_key,
)

def passage_generate_text(system_prompt, user_prompt):
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role":"system", "content":system_prompt},
            {"role": "user", "content": user_prompt}
            ]
    )
    return response.choices[0].message.content.strip()

def key_point_generate_text(system_prompt, user_prompt):
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role":"system", "content":system_prompt},
            {"role": "user", "content": user_prompt}
            ]
    )
    return response.choices[0].message.content.strip()


## 시스템 프롬프트

In [10]:
system_text = '''
당신은 대한민국 수학능력시험 국어영역 독서 과목의 지문을 출제하는 한국교육과정평가원 출제위원이다. 
고등학교 3학년 수준의 수험생을 평가할 수 있는 지문을 아래의 핵심 논점 및 난이도 요구사항, 작성 조건, 금지사항을 반영하여 작성하십시오.

**핵심 논점 및 난이도 요구사항**
- 주요 개념 간의 관계를 논리적으로 설명하고, 지문에 나타난 논지의 타당성을 검토할 것
- 동일한 화제에 대한 상반되거나 다양한 관점을 비교·분석하며, 각 관점의 타당성을 비판적으로 평가할 수 있도록 구성할 것
- 각 분야의 학문적 배경을 반영하되, 개념적 깊이를 확보하고, 전문 용어는 문맥 속에서 명확히 설명할 것
- 단순한 정보 전달이 아니라 수험생이 논리적 추론을 수행할 수 있도록 유도할 것

**작성 조건**  
-문장당 어절 수: 평균 17~25어절을 유지하되, 자연스러운 문장 흐름을 고려할 것
- 글자 수: 한글 기준 최소 1200자, 최대 1400자 
- 문체: 문어체 사용, 종결어미는 반드시 ‘-다.’ 사용, 문장은 반드시 완결성을 갖출 것
- 개념 설명 방식: 장·단점 나열 방식이 아닌 개념 간 관계 중심 설명
- 표현 반복 금지: 같은 단어 반복 대신 유의어나 대체어 활용
- 문법 준수: 맞춤법, 띄어쓰기, 주어-서술어 호응 고려
- 신뢰성 준수 : 일정한 기준을 유지하여 신뢰할 수 있는 결과 제공
- 인용 표기: 인용 문장은 “ ”, 인용된 어구는 ‘ ’ 사용

**금지 사항** 
- 모호하거나 중의적인 표현 사용 금지
- 문학 작품 생성 금지
- 허구적 사건이나 인물명 사용 금지
- 비문 생성 금지 - (가), (나), (다) 등의 기호 사용 금지
- 결론에서 전체 내용을 요약하거나 교훈 제시 금지
- 자극적이거나 선정적인 문체 사용 금지
- 특정 집단을 비하하거나 옹호하는 내용 또는 잘못된 고정관념을 유발하는 내용 작성 금지

위 조건을 철저히 준수하여 수학능력시험 국어영역 독서 과목 지문을 출제하십시오.'''

## 지문 생성

In [11]:
# 1. 디렉토리 내의 모든 지문 관련 .txt 파일을 불러오기
loader_passage = DirectoryLoader("./RAG자료/지문 관련", glob="*.txt", loader_cls=TextLoader)
documents_passage = loader_passage.load()

# 2. OpenAI 임베딩 모델 초기화
embeddings_model = OpenAIEmbeddings()

# 3. 지문 전체 문서 FAISS 인덱스 생성
texts_passage = [doc.page_content for doc in documents_passage]
metadata_passage = [doc.metadata for doc in documents_passage]
vector_db_passage_all = FAISS.from_texts(texts_passage, embeddings_model, metadatas=metadata_passage)


# 4. FAISS 인덱스 저장 (지문 전체 / 지문 필수 문서 따로 저장)
faiss_passage_all_path = "./faiss_index/faiss_index_passage_all"

vector_db_passage_all.save_local(faiss_passage_all_path)
print(f"지문 전체 문서 FAISS 벡터 DB 저장 완료: {faiss_passage_all_path}")


지문 전체 문서 FAISS 벡터 DB 저장 완료: ./faiss_index/faiss_index_passage_all


In [12]:
subject_query = "기술"
topic_query = "인공지능과 기계학습"

In [13]:
# 전체 문서 FAISS에서 유사한 문서 1개 검색
passage_guidelines = vector_db_passage_all.similarity_search(subject_query, k=1)

# 검색 결과 출력
for i, res in enumerate(passage_guidelines):
    print(f"\n[{i+1}] 검색된 문서:")
    print("-" * 50)
    print(res.page_content)
    print("-" * 50)
    print("📄 Metadata:", res.metadata)



[1] 검색된 문서:
--------------------------------------------------
기술 분야의 글이란?
- 산업 기술, 생활 기술 등 다양한 기술을 설명하는 글
- 과학 이론을 바탕으로 장치나 시스템의 원리, 작동 과정, 한계를 구체적으로 서술
- 포함되는 분야:
  - 전기,전자 공학 기술
  - 컴퓨터 공학 기술
  - 화학·생명과학과 결합된 공학 기술
  - 토목·건축 공학 기술

기술 분야의 글 읽기 방법
- 장치·시스템의 작동 원리를 단계별로 이해
- 구성 요소의 기능과 특징을 파악해 작동 원리 이해
- 낯선 용어의 개념을 정확히 파악하고 원리·방법과 연결

기술 분야의 출제 경향
- 실생활에서 접할 수 있는 장치·시스템 관련 내용 출제
- 전문적인 기술 내용을 다루는 경우가 많음
- 기술 발전과 관련된 정보 통신, 컴퓨터 공학, 영상 과학, 생명 공학 분야의 지문이 자주 출제됨


--------------------------------------------------
📄 Metadata: {'source': 'RAG자료\\지문 관련\\26수특_기술분야의 글이란.txt'}


### 지문 생성 프롬프트

In [14]:
passage_text = f'''
다음 {subject_query}와 {topic_query}를 바탕으로 대한민국 수학능력시험 국어영역 독서 과목 문제 풀이에 적합한 지문을 작성하세요.  
{subject_query} 분야에서 {topic_query}을 핵심 제재로 활용하여 논리적이고 구조적인 지문을 구성하십시오.  

[출제 기준]  
아래 "{subject_query}" 분야의 출제 경향 및 작성 원칙을 충실히 반영하여 지문을 작성하십시오.  

{passage_guidelines}  

**반영 방법**  
- "{subject_query} 분야의 글이란?" : 해당 분야의 글의 본질적인 특성과 주요 논의 대상을 명확히 포함할 것.  
- "{subject_query} 분야의 글 읽기 방법" :독자가 글의 구조와 전개 방식을 쉽게 이해할 수 있도록 논지를 명확히 전달할 것.  
- "{subject_query} 분야의 출제 경향" : 기존 출제 방식과 논리적 구성 원칙을 반영하여 지문을 구성할 것.  

이 기준을 기반으로, 독자가 개념을 명확히 이해하고 논리적 추론을 수행할 수 있도록 논리적인 전개 방식과 구조를 갖춘 지문을 작성하십시오.  
실제 수능 독서 지문과 유사한 형식과 난이도를 유지하십시오.  

'''

In [15]:
passage_result = passage_generate_text(system_text, passage_text)

In [16]:
passage_result

'인공지능(AI)과 기계 학습(ML)은 현대 기술 발전의 중심에 있다. 이 두 가지 기술은 여러 분야에서 혁신적인 변화를 주도하고 있다. 먼저, 인공지능은 인간의 지적 능력을 모방하고자 하며, 이를 위해 복잡한 데이터 분석과 의사 결정 과정을 처리한다. 반면, 기계 학습은 대량의 데이터를 훈련하여 패턴을 인식하고 예측 모델을 구축하는 데 초점을 맞춘다. 이 두 기술은 서로 밀접하게 연결되어 있으며, 인공지능의 여러 하위 분야 중 하나로서 기계 학습이 위치한다.\n\n기계 학습의 대표적인 방법에는 지도 학습과 비지도 학습이 있다. 지도 학습은 기존의 레이블이 있는 데이터를 기반으로 하여 새로운 데이터의 결과를 예측하는 데 사용된다. 예를 들어, 고객의 구매 이력을 통해 향후 구매 가능성을 예측하는 시스템이 이에 해당한다. 반대로 비지도 학습은 레이블이 없는 데이터에서 패턴이나 구조를 발견하는 데 목적을 둔다. 이는 고객 세분화를 통해 비슷한 성향의 집단을 나누는 작업에 활용될 수 있다.\n\n좀 더 구체적으로, 지원 벡터 기계(SVM) 나 인공 신경망(ANN)과 같은 기계 학습 알고리즘은 인공지능의 다양한 응용 분야에서 필수적인 역할을 한다. 예를 들어, 의료 영상에서 암 세포를 판별하거나 자율 주행 차량의 경로를 최적화하는 등의 작업에서 활용된다. 이러한 알고리즘은 방대한 데이터를 통해 스스로 학습하고 점점 더 정확한 결과를 도출하도록 설계되었다.\n\n그러나 이러한 기술의 발전에는 여러 가지 도전 과제가 따른다. 첫째, 데이터 편향 문제이다. 인공지능 시스템이 잘못된 데이터나 편향된 데이터를 학습할 경우, 불공정한 결과를 초래할 수 있다. 이 문제를 해결하기 위해서는 데이터 수집 과정에서의 정확성과 다양한 데이터셋의 활용이 필수적이다. 둘째, 개인정보 보호 문제도 주목해야 한다. 인공지능 시스템이 개인 데이터를 처리할 때, 데이터의 익명성을 유지하고 프라이버시를 보호하는 것이 중요한 쟁점으로 떠오르고 있다.\n\n여러 과학자와 연구자들은 인공지능과 기계 학습

In [17]:
type(passage_result)

str

## 논점 생성

In [37]:
key_points_prompt = f"""
다음은 한국교육과정평가원 스타일로 생성된 수능 독서 지문입니다.

[생성된 지문]
{passage_result}

이 지문에서 학생이 반드시 이해해야 할 핵심 논점 2~3개를 요약하세요.
각 논점은 1~2문장으로 정리하고, 출제 의도를 반영해야 합니다.

"""
key_points = key_point_generate_text(key_points_prompt)
key_points

'1. **기계학습의 원리와 활용**: 기계학습은 대량의 데이터를 분석하여 패턴을 인식하고 이를 바탕으로 예측이나 의사결정을 내리는 기술로, 다양한 분야에서 기존의 인간 작업을 자동화하고 향상시키는 역할을 합니다. 이 논점은 학생이 기계학습의 기본 원리와 실제 응용 사례를 이해하도록 돕는 데 중점을 둡니다.\n\n2. **기계학습의 한계와 주의점**: 기계학습은 데이터의 편향이나 부정확성 때문에 잘못된 결과를 초래할 수 있으며, 예기치 않은 상황 대응에 한계가 있습니다. 이 문제를 극복하기 위해서는 데이터의 품질을 관리하고 모델의 신뢰성을 지속적으로 평가하는 것이 필요합니다. 이 논점은 학생이 기계학습의 한계를 인식하고 이를 개선하기 위한 책임 있는 접근이 필요함을 이해하도록 유도합니다.\n\n3. **기계학습의 사회적 영향**: 인공지능과 기계학습은 현대 기술 발전의 중요한 요소로, 우리의 삶을 편리하고 효율적으로 만듭니다. 그러나 기술 적용에는 책임과 주의가 필요하며, 데이터의 질과 편향성을 지속적으로 모니터링하고 개선하여 기술이 긍정적인 사회적 영향을 미치도록 해야 합니다. 이 논점은 기술의 사회적 영향과 책임 있는 활용의 중요성을 학생이 이해하도록 돕습니다.'

## 문항 생성

In [12]:
# 1. 디렉토리 내의 모든 문제 관련 .txt 파일을 불러오기
loader_question = DirectoryLoader("./RAG자료/문제 관련", glob="*.txt", loader_cls=TextLoader)
documents_question = loader_question.load()

# 2. 필수 문서 목록 (문제 관련)
loader_required_question = DirectoryLoader("./RAG자료/문제 관련/공통", glob="*.txt", loader_cls=TextLoader)
required_documents_question = loader_required_question.load()

# 3. OpenAI 임베딩 모델 초기화
embeddings_model = OpenAIEmbeddings()

# 4.1 문제 전체 문서 FAISS 인덱스 생성
texts_question = [doc.page_content for doc in documents_question]
metadata_question = [doc.metadata for doc in documents_question]
vector_db_question_all = FAISS.from_texts(texts_question, embeddings_model, metadatas=metadata_question)

# 4.2 문제 필수 문서만 FAISS 인덱스 생성
if required_documents_question:
    required_texts_question = [doc.page_content for doc in required_documents_question]
    required_metadata_question = [doc.metadata for doc in required_documents_question]
    vector_db_question_required = FAISS.from_texts(required_texts_question, embeddings_model, metadatas=required_metadata_question)
else:
    vector_db_question_required = None

# 5. FAISS 인덱스 저장 (문항 전체 / 문항 필수 문서 따로 저장)
faiss_question_all_path = "./faiss_index/faiss_index_question_all"
faiss_question_required_path = "./faiss_index/faiss_index_question_required"

vector_db_question_all.save_local(faiss_question_all_path)
print(f"문제 전체 문서 FAISS 벡터 DB 저장 완료: {faiss_question_all_path}")

if vector_db_question_required:
    vector_db_question_required.save_local(faiss_question_required_path)
    print(f"문제 필수 문서 FAISS 벡터 DB 저장 완료: {faiss_question_required_path}")


문제 전체 문서 FAISS 벡터 DB 저장 완료: ./faiss_index/faiss_index_question_all
문제 필수 문서 FAISS 벡터 DB 저장 완료: ./faiss_index/faiss_index_question_required


In [48]:
# 전체 문서 FAISS에서 유사한 문서 3개 검색
results = vector_db_question_all.similarity_search(subject_query, k=3)

# 필수 문서 FAISS에서 모든 문서 가져오기
if vector_db_question_required:  # 필수 문서 인덱스가 존재하는 경우에만 추가
    required_results = vector_db_question_required.similarity_search(subject_query, k=2)  # 필수 문서 검색
else:
    required_results = []

# 중복 제거 (유사 문서 + 필수 문서 통합)
unique_results = {res.page_content: res for res in results}  # 전체 문서 검색 결과 저장

# 필수 문서 추가 (중복 방지)
for doc in required_results:
    unique_results.setdefault(doc.page_content, doc)  # 중복되지 않는 경우만 추가

# 최종 검색 결과 리스트 (최대 5개)
question_guildlines = list(unique_results.values())[:5]

# 검색 결과 출력
for i, res in enumerate(question_guildlines):
    print(f"\n[{i+1}] 검색된 문서:")
    print("-" * 50)
    print(res.page_content)
    print("-" * 50)
    print("📄 Metadata:", res.metadata)



[1] 검색된 문서:
--------------------------------------------------
- 기술의 핵심 원리나 방법을 <보기>의 그림으로 제시하여, 장치나 시스템의 작동 원리에 대해 추론을 할 수 있는지를 평가히는 문항이 자주 출제되고 있다.
--------------------------------------------------
📄 Metadata: {'source': 'RAG자료\\문제 관련\\26수특_기술분야의 출제경향_문항.txt'}

[2] 검색된 문서:
--------------------------------------------------
지문에서 설명한 개념이나 원리, 문제의 해결 방안 등을 실제 현상이나 상황에 적용하여 추론적 사고를 수행할 수 있는지를 묻는 문항이 자주 출제되고 있다.
--------------------------------------------------
📄 Metadata: {'source': 'RAG자료\\문제 관련\\26수특_사회문화분야의 출제경향_문항.txt'}

[3] 검색된 문서:
--------------------------------------------------
- 지문에 제시되어 있는 핵심 정보를 근거로 삼아 추론을 할 수 있는지를 평가하는 문항이 매번 출제되고 있다.
--------------------------------------------------
📄 Metadata: {'source': 'RAG자료\\문제 관련\\26수특_과학분야의 출제경향_문항.txt'}

[4] 검색된 문서:
--------------------------------------------------
- 인문학·사회학·자연과학· 기술 공학·예술·생활 분야의 다양한 제재를 다룬다.
- 독서의 원리와 방법에 대한 지식과 아울러 어휘력, 사실적·추론적·비판적·창의적 사고력 등을 측정할 수 있는 문항을 출제한다.
- 설명문·논설문·서사문·보고서·생활문 등 다양한 유형의 글을 활용하여 출제하되, 지문

In [50]:
question_type =""

#부정형/정답형 처리
if "않은" in question_type or "않는" in question_type:
    narrative_style = "부정형"
else:
    narrative_style = "정답형"

In [51]:
user_prompt = f"""
    다음은 문제와 선택지를 작성할 때 반드시 고려해야 할 기준이다. 문제와 선택지를 작성하는 지침에 따라 내용을 참고하여 정확히 반영하시오:
    아래 지문을 바탕으로 학생의 이해력, 논리적 추론 능력, 비판적 사고력을 평가할 수 있는 5지선다형 객관식 문항을 1개 작성하시오.

    [지문]
    {passage_result}

    [핵심 논점(출제 의도)]
    다음은 지문에서 강조되었던 핵심 논점입니다. 문항 출제 시 반드시 아래 논점을 바탕으로 학생의 이해 및 사고력을 평가할 수 있도록 하세요.
    {key_points}

    [출제 방향 설정]
    - 반드시 위 '핵심 논점'에 근거하여 문제를 출제하십시오.
    - 다음은 "{subject_query}" 관련 수학능력시험 국어영역 독서과목 출제 경향 가이드라인입니다.
      이 기준을 충실히 반영하여 문제를 작성하십시오.
		{question_guildlines}
    - 출제 오류가 발생하지 않도록, 선택지는 모두 지문 내용이나 논리와 명백히 연결되어야 합니다.
    
        [금지 사항]
        - 지문의 내용과 관련 없는 문제 생성 금지.
        - 비논리적이거나 두 개 이상의 답이 나올 수 있는 문제 생성 금지.
        - 문제와 정답이 명확하지 않은 경우 생성 금지.
        - 지문의 한 정보를 읽고 풀 수 있는 선택지가 2개 이상 있으면 안 됨

        [작성 조건]
        - 하나의 선택지에서는 하나의 정보만 물어볼 것.
        - 지문에 쓰인 명사는 풀어쓰거나 비슷한 표현으로 바꾸지 않고 그대로 쓸 것.
		    - 선택지의 문장은 길이에 따라 짧은 문장에서 긴 문장 순으로 배열하십시오.
		    - 선택지의 길이 정렬이 지켜지지 않으면 출제 오류로 간주합니다.
		    - 예시:
		      1. 밑줄 긋기는 독자의 기억을 돕는다.
		      2. 무분별한 밑줄 긋기는 독서 흐름을 방해할 수 있다.
		      3. 특정 정보를 강조하여 시각적 주의를 기울이도록 한다.
		      4. 너무 많은 정보를 표시하면 중요한 내용을 파악하기 어려워질 수 있다.
		      5. 효과적인 밑줄 긋기 방법은 핵심 정보만 표시하고 과도한 사용을 자제하는 것이다.

    [문제 생성 출력 형식]
    [문제 유형]
    문항 유형 사실적 읽기/추론적 읽기/비판적 읽기 중 1개
    서술 방식 {narrative_style}
    
    [논점]
    {key_points}
    
    [질문]
    {question_type}
    
    [선택지]
    1. (내용)
    2. (내용)
    3. (내용)
    4. (내용)
    5. (내용)
    
    정답: X번
    
    문제 해설: (정답의 근거와 오답이 틀린 이유를 포함한 상세 해설 최소 100자 최대 200자)
    
    
    
    """


In [26]:
question_data = generate_text(user_prompt)

'질문: 지도 학습과 비지도 학습의 차이점으로 올바르게 설명된 것은 무엇인가?\n\n///\n1. 지도 학습은 정답이 없는 데이터 세트를 사용하여 패턴을 인식한다.\n///\n2. 비지도 학습은 정답이 포함된 데이터 세트를 사용하여 모델의 정확성을 향상시킨다.\n///\n3. 지도 학습은 정답을 포함한 데이터 세트를 사용하여 문제를 해결한다.\n///\n4. 비지도 학습에서는 데이터의 군집화와 분류가 불가능하다.\n///\n5. 지도 학습에서는 정답이 없는 데이터 세트를 사용하여 잠재 구조를 찾는다.\n///\n\n정답: 3번\n\n///\n해설: 3번 선택지는 지도 학습의 특성을 정확히 설명하고 있다. 지도 학습에서는 정답(레이블)이 포함된 데이터 세트를 사용하여 학습하고, 이를 통해 모델의 정확성을 향상시킨다. 1번과 5번 선택지는 비지도 학습에 대한 설명으로, 지도 학습이 아닌 비지도 학습에서는 정답이 없는 데이터 세트를 활용하여 패턴을 인식하고 잠재 구조를 찾는다. 2번 선택지는 지도 학습과 비지도 학습의 개념을 혼동하고 있어 틀렸다. 4번 선택지 또한 비지도 학습의 특징을 잘못 이해한 것이다. 비지도 학습은 데이터를 군집화하거나 분류할 수 있는 잠재 구조를 찾는 데 중점을 둔다.'

In [ ]:
question_data